In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk", font="DejaVu Sans")

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

RESULTS_FILE = "first_crossing_results.csv"

results = pd.read_csv(RESULTS_FILE)

results["resolved_label"] = results["resolved_yes"].map({False: "Resolved NO", True: "Resolved YES"})
results["volume_bin"] = pd.cut(
    results["volume_fp"],
    bins=[0, 10000, 50000, 100000, np.inf],
    labels=["10k–50k", "50k–100k", "100k–250k", "250k+"],
    include_lowest=True
)
results["start_bin"] = pd.cut(
    results["min_bid"],
    bins=[0, 0.05, 0.10, 0.20, 1.0],
    labels=["0–5¢", "5–10¢", "10–20¢", "20%+"],
    include_lowest=True
)

summary1 = results.groupby("resolved_label", observed=True)["crossed_5pct"].agg(["count", "mean"]).reindex(["Resolved NO", "Resolved YES"])
summary1.to_csv(OUT_DIR / "table1_cross_rates.csv")

summary2 = results.groupby(["resolved_label", "volume_bin"], observed=True)["crossed_5pct"].agg(["count", "mean"]).reset_index()
summary2.to_csv(OUT_DIR / "table2_volume_bins.csv", index=False)

summary3 = results.groupby(["start_bin", "resolved_label"], observed=True)["crossed_5pct"].agg(["count", "mean"]).reset_index()
summary3.to_csv(OUT_DIR / "table3_start_bins.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6))
palette = ["#2C7FB8", "#F28E2B"]
vals = summary1["mean"].values
counts = summary1["count"].values
ax.bar(summary1.index, vals, color=palette, edgecolor="black", linewidth=1.0)
ax.set_title("Crossing Below 5¢ by Final Resolution", pad=12, weight="bold")
ax.set_ylabel("Share crossing below 5¢")
ax.set_xlabel("")
ax.set_ylim(0, 1)
for i, (v, n) in enumerate(zip(vals, counts)):
    ax.text(i, v + 0.02, f"{v:.1%}\n(n={int(n)})", ha="center", va="bottom", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure1_cross_by_resolution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(results["min_bid"].clip(0, 1), bins=40, color="#4C78A8", alpha=0.9, edgecolor="white")
ax.axvline(0.05, color="#D62728", linestyle="--", linewidth=2, label="5¢ threshold")
ax.set_title("Distribution of Minimum Bid Price", pad=12, weight="bold")
ax.set_xlabel("Minimum observed bid")
ax.set_ylabel("Number of markets")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure2_min_bid_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

vol_summary = results.groupby("volume_bin", observed=True)["crossed_5pct"].agg(["mean", "count"]).reset_index()
fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(vol_summary["volume_bin"].astype(str), vol_summary["mean"], color="#59A14F", edgecolor="black", linewidth=1.0)
ax.set_title("Crossing Rate by Volume Bucket", pad=12, weight="bold")
ax.set_xlabel("Volume bucket")
ax.set_ylabel("Share crossing below 5¢")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=25)
for i, row in vol_summary.iterrows():
    ax.text(i, row["mean"] + 0.02, f"{row['mean']:.1%}\n(n={int(row['count'])})", ha="center", va="bottom", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "figure3_cross_by_volume.png", dpi=300, bbox_inches="tight")
plt.close(fig)

ct = pd.crosstab(results["crossed_5pct"], results["resolved_yes"], normalize="index")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    ct,
    annot=True,
    fmt=".2f",
    cmap="mako",
    cbar_kws={"label": "Row share"},
    ax=ax,
    linewidths=0.5,
    linecolor="white"
)
ax.set_title("Resolution Given Crossing Below 5¢", pad=12, weight="bold")
ax.set_xlabel("Resolved YES")
ax.set_ylabel("Crossed below 5¢")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure4_resolution_heatmap.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:")
for p in sorted(OUT_DIR.glob("*")):
    print(p.name)

Saved:
figure1_cross_by_resolution.png
figure2_min_bid_distribution.png
figure3_cross_by_volume.png
figure4_resolution_heatmap.png
table1_cross_rates.csv
table2_volume_bins.csv
table3_start_bins.csv
